<a href="https://colab.research.google.com/github/ericyoc/3d_printer_sidechannel_poc/blob/main/3d_printer_sidechannel_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Modal Side-Channel Attack and Defense Evaluation on 3D Printers
### Acoustic + Vibration Fusion Against Active Motor Noise Cancellation

**Dataset:** Madamopoulos & Tsoutsos (2024). *3D Printer Audio and Vibration Side Channel Dataset.*  
Zenodo. https://doi.org/10.5281/zenodo.13329934

**Experimental Design:**  
This notebook empirically evaluates four conditions in sequence:

| Phase | Description |
|---|---|
| **Pre-Attack Baseline** | Signal discriminability before any attack model is applied |
| **Post-Attack** | Classification accuracy: Acoustic-only, Vibration-only, Fused |
| **Post-Defense** | Simulated Active Motor Noise Cancellation applied to acoustic channel |
| **Defense Evaluation** | Does fusion defeat the countermeasure? |

**Research Question:** Does multi-modal fusion of acoustic and vibration side-channels defeat Active Motor Noise Cancellation (AMNC) in modern FDM 3D printers?

---
**Setup Instructions:**
1. Download the dataset from https://zenodo.org/records/13329934
2. Upload the extracted folder to your Google Drive at: `MyDrive/3dprinter_sidechannel/`
3. Run all cells in order

In [ ]:
# Install required libraries
!pip install -q librosa scikit-learn pandas numpy matplotlib seaborn scipy pydub

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
OUTPUT_DIR = '/content/drive/MyDrive/3dprinter_sidechannel/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Dataset base: {DRIVE_BASE}')
print(f'Results will be saved to: {OUTPUT_DIR}')

Mounted at /content/drive
Dataset base: /content/drive/MyDrive/3dprinter_sidechannel
Results will be saved to: /content/drive/MyDrive/3dprinter_sidechannel/results


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import warnings
import glob
import json
from pathlib import Path
from scipy import signal as scipy_signal
from scipy.stats import zscore

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# ── Publication figure style ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
})

# Single-column article width: 3.5 inches; double-column: 7.0 inches
COL1 = 3.5
COL2 = 7.0

# Colorblind-safe palette
PALETTE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7']

print('Environment ready.')

Environment ready.


In [ ]:
# ── Auto-download Zenodo dataset to Google Drive ──────────────────────────────
!pip install -q zenodo-get

import os
DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Download directly from Zenodo into the Drive folder
os.chdir(DRIVE_BASE)
!zenodo_get 13329934

# Show what was downloaded
for root, dirs, files in os.walk(DRIVE_BASE):
    depth = root.replace(DRIVE_BASE, '').count(os.sep)
    if depth > 3: continue
    print('  ' * depth + os.path.basename(root) + '/')
    for f in files[:5]:
        print('  ' * (depth+1) + f)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 7.1 MB/s eta 0:00:00
INFO: Output directory: /content/drive/MyDrive/3dprinter_sidechannel
INFO: Title: 3D printer audio and vibration side channels
INFO: Total size: 10.8 GB
INFO: Number of files: 16
INFO: 05_2keys.zip is already downloaded correctly.
INFO: 07_NIST_additive_test.zip is already downloaded correctly.
INFO: 08_ASTM.zip is already downloaded correctly.
INFO: 10_Triple_Helix.zip is already downloaded correctly.
INFO: Printing Videos.zip is already downloaded correctly.
INFO: 12_retraction_test.zip is already downloaded correctly.
INFO: 11_CaliCat.zip is already downloaded correctly.
INFO: Noise Recordings.zip is already downloaded correctly.
INFO: 04_key_steps.zip is already downloaded

In [ ]:
import zipfile
import os

DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'

# Extract all zips in place
for f in os.listdir(DRIVE_BASE):
    if f.endswith('.zip'):
        zip_path = os.path.join(DRIVE_BASE, f)
        extract_to = os.path.join(DRIVE_BASE, f.replace('.zip', ''))
        print(f'Extracting {f}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print(f'  Done -> {extract_to}')

# Show resulting structure
for root, dirs, files in os.walk(DRIVE_BASE):
    depth = root.replace(DRIVE_BASE, '').count(os.sep)
    if depth > 3: continue
    print('  ' * depth + os.path.basename(root) + '/')
    for f in files[:3]:
        print('  ' * (depth+1) + f)

Extracting 05_2keys.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/05_2keys
Extracting 07_NIST_additive_test.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/07_NIST_additive_test
Extracting 08_ASTM.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/08_ASTM
Extracting 10_Triple_Helix.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/10_Triple_Helix
Extracting Printing Videos.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/Printing Videos
Extracting 12_retraction_test.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/12_retraction_test
Extracting 11_CaliCat.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/11_CaliCat
Extracting Noise Recordings.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/Noise Recordings
Extracting 04_key_steps.zip...
  Done -> /content/drive/MyDrive/3dprinter_sidechannel/04_key_steps
Extracting 06_Autodesk_kickstarter_FDM_test.zip...
  Done -> /content/drive/MyDri

In [ ]:
# ── Discover dataset files ────────────────────────────────────────────────────
def discover_dataset(base_path):
    records = []
    base = Path(base_path)
    audio_exts = {'.mp3', '.caf', '.wav', '.m4a'}
    vibr_exts  = {'.csv'}

    # Skip these non-object folders
    skip_folders = {'results', 'Printing Videos', 'Noise Recordings',
                    'Artifacts', '__MACOSX'}

    # Each object folder is e.g. 01_key_easy/1_key_easy/Bambu_A1mini/
    for obj_zip_dir in sorted(base.iterdir()):
        if not obj_zip_dir.is_dir(): continue
        if obj_zip_dir.name in skip_folders: continue
        if obj_zip_dir.name.endswith('.zip'): continue

        # Step into the single nested subfolder (e.g. 1_key_easy/)
        nested = [d for d in obj_zip_dir.iterdir() if d.is_dir()]
        if not nested: continue
        obj_root = nested[0]
        obj_id = obj_zip_dir.name   # e.g. "01_key_easy"

        # Now find Bambu_A1mini and Bambu_P1P subdirs
        for printer_dir in sorted(obj_root.iterdir()):
            if not printer_dir.is_dir(): continue
            if printer_dir.name == 'desktop.ini': continue
            printer_name = printer_dir.name  # Bambu_A1mini or Bambu_P1P

            # Walk all subdirs for audio + vibration files
            audio_files = list(printer_dir.rglob('*'))
            audio_files = [f for f in audio_files
                          if f.suffix.lower() in audio_exts]
            vibr_files  = [f for f in printer_dir.rglob('*')
                          if f.suffix.lower() in vibr_exts]

            if not audio_files:
                continue

            for af in audio_files:
                # Match vibration file by closest name if multiple exist
                vf = vibr_files[0] if vibr_files else None
                records.append({
                    'printer':     printer_name,
                    'object_id':   obj_id,
                    'audio_path':  str(af),
                    'vibr_path':   str(vf) if vf else None,
                    'amnc_active': 'a1mini' in printer_name.lower()
                })

    return pd.DataFrame(records)


catalog = discover_dataset(DRIVE_BASE)
print(f'Files discovered: {len(catalog)}')
if len(catalog) > 0:
    print(catalog.groupby(['printer','amnc_active']).size().reset_index(name='count').to_string(index=False))
    print('\nSample entries:')
    print(catalog[['printer','object_id','amnc_active']].head(10).to_string(index=False))
else:
    print('Still empty — paste the output of this:')
    print('  list(Path(DRIVE_BASE).rglob("*.mp3"))[:5]')
    print('  list(Path(DRIVE_BASE).rglob("*.caf"))[:5]')

Files discovered: 144
     printer  amnc_active  count
Bambu_A1mini         True     72
   Bambu_P1P        False     72

Sample entries:
     printer   object_id  amnc_active
Bambu_A1mini 01_key_easy         True
Bambu_A1mini 01_key_easy         True
Bambu_A1mini 01_key_easy         True
Bambu_A1mini 01_key_easy         True
Bambu_A1mini 01_key_easy         True
Bambu_A1mini 01_key_easy         True
   Bambu_P1P 01_key_easy        False
   Bambu_P1P 01_key_easy        False
   Bambu_P1P 01_key_easy        False
   Bambu_P1P 01_key_easy        False


In [ ]:
# ── AMNC simulation: notch filter bank on motor resonance band ────────────────
def simulate_amnc(y, sr, notch_freqs=None, Q=30):
    """
    Apply a bank of notch filters to simulate Active Motor Noise Cancellation.
    notch_freqs: list of centre frequencies (Hz) to suppress.
    Q: quality factor — higher = narrower notch.
    """
    if notch_freqs is None:
        # Typical FDM stepper motor resonances: fundamental + harmonics
        notch_freqs = [120, 180, 240, 300, 360]
    y_filtered = y.copy()
    for f0 in notch_freqs:
        if f0 < sr / 2:            # only if below Nyquist
            b, a = scipy_signal.iirnotch(f0, Q, sr)
            y_filtered = scipy_signal.filtfilt(b, a, y_filtered)
    return y_filtered


# ── Acoustic feature extraction ───────────────────────────────────────────────
def extract_acoustic_features(audio_path, apply_amnc=False, n_mfcc=13):
    """
    Returns a 1-D feature vector from an audio file.
    apply_amnc=True simulates the AMNC countermeasure.
    """
    try:
        y, sr = librosa.load(audio_path, sr=22050, mono=True)
    except Exception:
        return None

    if apply_amnc:
        y = simulate_amnc(y, sr)

    mfcc        = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    cent        = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw          = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff     = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr         = librosa.feature.zero_crossing_rate(y)

    feats = np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        cent.mean(axis=1), cent.std(axis=1),
        bw.mean(axis=1),   bw.std(axis=1),
        rolloff.mean(axis=1),
        zcr.mean(axis=1)
    ])
    return feats


# ── Vibration feature extraction ──────────────────────────────────────────────
def extract_vibration_features(vibr_path):
    """
    Returns a 1-D feature vector from a vibration CSV.
    Expected columns: timestamp, x, y, z  (or similar).
    """
    try:
        df = pd.read_csv(vibr_path)
    except Exception:
        return None

    # Flexibly identify axis columns
    axis_cols = [c for c in df.columns if c.lower() in ['x','y','z',
                 'accel_x','accel_y','accel_z','ax','ay','az']]
    if not axis_cols:
        # Fall back: use all numeric columns except timestamp
        axis_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:3]

    feats = []
    for col in axis_cols:
        v = df[col].dropna().values.astype(float)
        if len(v) < 10:
            continue
        fft_mag = np.abs(np.fft.rfft(v))
        feats.extend([
            v.mean(),
            v.std(),
            np.sqrt(np.mean(v**2)),                    # RMS
            v.max() - v.min(),                          # peak-to-peak
            np.fft.rfftfreq(len(v))[np.argmax(fft_mag)] # dominant freq
        ])

    return np.array(feats) if feats else None


print('Feature extraction functions defined.')

Feature extraction functions defined.


In [ ]:
# ── Copy dataset to local disk first (prevents Drive RAM crash) ───────────────
import shutil
LOCAL_BASE = '/content/3dp_local'

if not os.path.exists(LOCAL_BASE):
    print('Copying dataset to local disk...')
    shutil.copytree(DRIVE_BASE, LOCAL_BASE,
                    ignore=shutil.ignore_patterns('results', '*.zip'))
    print('Copy complete.')
else:
    print('Local copy already exists.')

catalog['audio_path'] = catalog['audio_path'].str.replace(DRIVE_BASE, LOCAL_BASE)
catalog['vibr_path']  = catalog['vibr_path'].fillna('').str.replace(DRIVE_BASE, LOCAL_BASE)
catalog['vibr_path']  = catalog['vibr_path'].replace('', None)


# ── RAM-safe feature extractors ───────────────────────────────────────────────
def simulate_amnc(y, sr, notch_freqs=None, Q=30):
    if notch_freqs is None:
        notch_freqs = [120, 180, 240, 300, 360]
    y_filtered = y.copy()
    for f0 in notch_freqs:
        if f0 < sr / 2:
            b, a = scipy_signal.iirnotch(f0, Q, sr)
            y_filtered = scipy_signal.filtfilt(b, a, y_filtered)
    return y_filtered


def extract_acoustic_features(audio_path, apply_amnc=False, n_mfcc=13):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True, duration=30)
    except Exception:
        return None
    if apply_amnc:
        y = simulate_amnc(y, sr)
    mfcc    = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    cent    = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw      = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr     = librosa.feature.zero_crossing_rate(y)
    return np.concatenate([
        mfcc.mean(axis=1), mfcc.std(axis=1),
        cent.mean(axis=1), cent.std(axis=1),
        bw.mean(axis=1),   bw.std(axis=1),
        rolloff.mean(axis=1),
        zcr.mean(axis=1)
    ])


def extract_vibration_features(vibr_path):
    try:
        df = pd.read_csv(vibr_path)
    except Exception:
        return None
    axis_cols = [c for c in df.columns if c.lower() in
                 ['x','y','z','accel_x','accel_y','accel_z','ax','ay','az']]
    if not axis_cols:
        axis_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:3]
    feats = []
    for col in axis_cols:
        v = df[col].dropna().values.astype(float)
        if len(v) < 10:
            continue
        fft_mag = np.abs(np.fft.rfft(v))
        feats.extend([
            v.mean(),
            v.std(),
            np.sqrt(np.mean(v**2)),
            v.max() - v.min(),
            np.fft.rfftfreq(len(v))[np.argmax(fft_mag)]
        ])
    return np.array(feats) if feats else None


# ── Build feature matrix ──────────────────────────────────────────────────────
print('Extracting features...')
records = []

for i, row in catalog.iterrows():
    if i % 10 == 0:
        print(f'  {i}/{len(catalog)}...')

    a_feats   = extract_acoustic_features(row['audio_path'], apply_amnc=False)
    a_feats_d = extract_acoustic_features(row['audio_path'], apply_amnc=True)
    v_feats   = extract_vibration_features(row['vibr_path']) \
                if pd.notna(row['vibr_path']) else None

    if a_feats is None:
        print(f'  Skipped: {row["audio_path"]}')
        continue

    records.append({
        'label':                   row['object_id'],
        'printer':                 row['printer'],
        'amnc_hw':                 row['amnc_active'],
        'acoustic_feats':          a_feats,
        'acoustic_feats_defended': a_feats_d,
        'vibr_feats':              v_feats,
    })

print(f'Done. {len(records)} samples across {catalog["object_id"].nunique()} objects.')

Copying dataset to local disk...
Copy complete.
Extracting features...
  0/144...
  10/144...
  20/144...
  30/144...
  40/144...
  50/144...
  60/144...
  70/144...
  80/144...
  90/144...
  100/144...
  110/144...
  120/144...
  130/144...
  140/144...
Done. 144 samples across 12 objects.


In [ ]:
# ── Prepare arrays ────────────────────────────────────────────────────────────
labels = np.array([r['label'] for r in records])
le     = LabelEncoder()
y_enc  = le.fit_transform(labels)
n_cls  = len(le.classes_)

X_audio   = np.vstack([r['acoustic_feats'] for r in records])
X_audio_d = np.vstack([r['acoustic_feats_defended'] for r in records])

# Vibration: some records may lack vibration data — handle gracefully
vibr_mask = np.array([r['vibr_feats'] is not None for r in records])
X_vibr_all = np.full((len(records), 15), np.nan)
for i, r in enumerate(records):
    if r['vibr_feats'] is not None:
        v = r['vibr_feats']
        X_vibr_all[i, :len(v)] = v[:15]

X_vibr = X_vibr_all

# Fused: concatenate audio + vibration, impute missing vibration with 0
X_vibr_imp = np.nan_to_num(X_vibr_all)
X_fused    = np.hstack([X_audio, X_vibr_imp])
X_fused_d  = np.hstack([X_audio_d, X_vibr_imp])

print(f'Acoustic feature dim : {X_audio.shape[1]}')
print(f'Vibration feature dim: {X_vibr_imp.shape[1]}')
print(f'Fused feature dim    : {X_fused.shape[1]}')
print(f'Samples              : {len(records)}')

Acoustic feature dim : 32
Vibration feature dim: 15
Fused feature dim    : 47
Samples              : 144


In [ ]:
# ── Classification utility ────────────────────────────────────────────────────
def evaluate_classifier(X, y, label, cv_folds=5):
    """
    Run stratified k-fold CV and return a metrics dict.
    Uses a Pipeline: StandardScaler + RandomForestClassifier.
    """
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(
                       n_estimators=200,
                       max_depth=None,
                       random_state=42,
                       n_jobs=-1))
    ])

    cv   = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    y_pred = cross_val_predict(pipe, X, y, cv=cv)

    return {
        'Condition':  label,
        'Accuracy':   round(accuracy_score(y, y_pred) * 100, 2),
        'F1 (macro)': round(f1_score(y, y_pred, average='macro') * 100, 2),
        'Precision':  round(precision_score(y, y_pred, average='macro') * 100, 2),
        'Recall':     round(recall_score(y, y_pred, average='macro') * 100, 2),
        'y_pred':     y_pred
    }


print('Running Phase 2 classifiers (post-attack, no defense)...')
res_audio  = evaluate_classifier(X_audio,    y_enc, 'Acoustic-Only (Pre-Defense)')
res_vibr   = evaluate_classifier(X_vibr_imp, y_enc, 'Vibration-Only')
res_fused  = evaluate_classifier(X_fused,    y_enc, 'Fused (Pre-Defense)')

print('Phase 2 complete.')

Running Phase 2 classifiers (post-attack, no defense)...
Phase 2 complete.


In [ ]:
print('Running Phase 3 classifiers (post-defense — AMNC applied)...')
res_audio_d = evaluate_classifier(X_audio_d, y_enc, 'Acoustic-Only (Post-Defense)')
res_fused_d = evaluate_classifier(X_fused_d, y_enc, 'Fused (Post-Defense)')
# Vibration-only is unchanged by AMNC — reuse res_vibr
print('Phase 3 complete.')

Running Phase 3 classifiers (post-defense — AMNC applied)...
Phase 3 complete.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LEAKAGE AUDIT + LEAKAGE-FREE RE-EVALUATION (vibration channel)
# Run AFTER X_vibr_imp / X_audio / X_fused / y_enc / records exist.
#
# Root cause: discover_dataset() matched every audio file in a printer folder to
# vibr_files[0], so all ~6 recordings of each (object, printer) share an IDENTICAL
# vibration vector. Plain StratifiedKFold(shuffle=True) then splits identical rows
# across folds -> the model memorizes them -> spurious 100%. This cell measures the
# duplication and re-scores with grouping that keeps identical rows together.
# ══════════════════════════════════════════════════════════════════════════════
import warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (StratifiedKFold, GroupKFold,
                                     LeaveOneGroupOut, cross_val_predict)
from sklearn.metrics import accuracy_score, f1_score

assert 'X_vibr_imp' in globals(), "Build the feature matrices first."
y      = y_enc
n_cls  = len(np.unique(y))
chance = 100.0 / n_cls

# ── 1. Duplication audit ──────────────────────────────────────────────────────
Xv   = np.nan_to_num(X_vibr_imp).astype(float)
keys = [hash(r.round(6).tobytes()) for r in Xv]          # identical rows -> same key
_, vib_group = np.unique(keys, return_inverse=True)      # group id per row
n_unique = len(set(keys))

print("="*64)
print("VIBRATION DUPLICATION AUDIT")
print("="*64)
print(f"Total samples              : {len(Xv)}")
print(f"DISTINCT vibration vectors : {n_unique}")
print(f"Duplicate rows             : {len(Xv) - n_unique} of {len(Xv)}")
print(f"Median copies per vector   : {int(pd.Series(vib_group).value_counts().median())}")
if n_unique <= 2 * n_cls + 2:
    print("=> Consistent with ONE vibration file per (object, printer).")

# ── 2. Three CV regimes: leaky vs leakage-free ────────────────────────────────
def rf():
    return Pipeline([('sc', StandardScaler()),
                     ('rf', RandomForestClassifier(n_estimators=200,
                                                   random_state=42, n_jobs=-1))])

def score(X, cv, groups=None):
    yp = cross_val_predict(rf(), np.nan_to_num(X).astype(float), y,
                           cv=cv, groups=groups)
    return round(accuracy_score(y, yp) * 100, 2)

skf     = StratifiedKFold(5, shuffle=True, random_state=42)          # leaky (original)
gkf     = GroupKFold(n_splits=min(5, n_unique))                      # group identical rows
printer = np.array([r.get('printer', '?') for r in records])
logo    = LeaveOneGroupOut()                                         # train 1 printer, test other

rows = []
for name, X in [('Acoustic', X_audio), ('Vibration', X_vibr_imp), ('Fused', X_fused)]:
    rec = {'Channel': name,
           'Leaky StratKFold (%)':   score(X, skf),
           'Grouped (no leak) (%)':  score(X, gkf, groups=vib_group)}
    if len(set(printer)) > 1:
        rec['Leave-1-printer-out (%)'] = score(X, logo, groups=printer)
    rows.append(rec)

print("\n" + "="*64)
print(f"LEAKAGE-FREE RE-EVALUATION    (random chance = {chance:.2f}%)")
print("="*64)
print(pd.DataFrame(rows).to_string(index=False))
print("""
Reading this table:
  • 'Leaky' reproduces the manuscript's 100% (identical rows split across folds).
  • 'Grouped' keeps identical vibration vectors together — the honest estimate.
  • 'Leave-1-printer-out' is the true cross-device attack (matches your transfer
    table). If vibration accuracy collapses toward chance in the right two columns,
    the within-printer 100% was duplicate-row memorization, not geometry leakage.
""")

VIBRATION DUPLICATION AUDIT
Total samples              : 144
DISTINCT vibration vectors : 24
Duplicate rows             : 120 of 144
Median copies per vector   : 6
=> Consistent with ONE vibration file per (object, printer).

LEAKAGE-FREE RE-EVALUATION    (random chance = 8.33%)
  Channel  Leaky StratKFold (%)  Grouped (no leak) (%)  Leave-1-printer-out (%)
 Acoustic                 11.11                    0.0                     6.94
Vibration                100.00                    0.0                    12.50
    Fused                100.00                    0.0                    18.06

Reading this table:
  • 'Leaky' reproduces the manuscript's 100% (identical rows split across folds).
  • 'Grouped' keeps identical vibration vectors together — the honest estimate.
  • 'Leave-1-printer-out' is the true cross-device attack (matches your transfer
    table). If vibration accuracy collapses toward chance in the right two columns,
    the within-printer 100% was duplicate-row memori

In [ ]:
import hashlib, pandas as pd
from pathlib import Path

BASE = Path('/content/3dp_local')                       # or DRIVE_BASE
if not BASE.exists(): BASE = Path('/content/drive/MyDrive/3dprinter_sidechannel')
skip = {'results','Printing Videos','Noise Recordings','Artifacts','__MACOSX'}

def md5(p):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(1<<20), b''): h.update(c)
    return h.hexdigest()

rows = []
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in skip or obj.name.endswith('.zip'): continue
    nested = [d for d in obj.iterdir() if d.is_dir()]
    if not nested: continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir(): continue
        csvs  = [f for f in pr.rglob('*') if f.suffix.lower()=='.csv']
        audio = [f for f in pr.rglob('*') if f.suffix.lower() in {'.mp3','.caf','.wav','.m4a'}]
        distinct = len({md5(c) for c in csvs}) if csvs else 0
        rows.append({'object':obj.name,'printer':pr.name,
                     'n_audio':len(audio),'n_vibr_csv':len(csvs),
                     'n_DISTINCT_vibr':distinct})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nTotal distinct vibration files: {df['n_DISTINCT_vibr'].sum()}")
print("If n_DISTINCT_vibr ≈ n_audio per row → your within-printer attack is RECOVERABLE.")
print("If n_DISTINCT_vibr == 1 per row → the data genuinely has one recording per class/printer.")

                          object      printer  n_audio  n_vibr_csv  n_DISTINCT_vibr
                     01_key_easy Bambu_A1mini        6           6                6
                     01_key_easy    Bambu_P1P        6           6                6
                   02_key_medium Bambu_A1mini        6           6                6
                   02_key_medium    Bambu_P1P        6           6                6
                     03_key_hard Bambu_A1mini        6           6                6
                     03_key_hard    Bambu_P1P        6           6                6
                    04_key_steps Bambu_A1mini        6           6                6
                    04_key_steps    Bambu_P1P        6           6                6
                        05_2keys Bambu_A1mini        6           6                6
                        05_2keys    Bambu_P1P        6           6                6
06_Autodesk_kickstarter_FDM_test Bambu_A1mini        6           6          

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path('/content/3dp_local')
if not BASE.exists(): BASE = Path('/content/drive/MyDrive/3dprinter_sidechannel')
AUD = {'.mp3','.caf','.wav','.m4a'}
EXPECTED = ['01_key_easy','02_key_medium','03_key_hard','04_key_steps','05_2keys',
            '06_Autodesk_kickstarter_FDM_test','07_NIST_additive_test','08_ASTM',
            '09_All_in_1','10_Triple_Helix','11_CaliCat','12_retraction_test']

print(f"Scanning: {BASE}\n")
rows = []
for obj in EXPECTED:
    od = BASE / obj
    a = p = 0
    if od.is_dir():
        nested = [d for d in od.iterdir() if d.is_dir()]
        root = nested[0] if nested else od
        for pr in root.iterdir():
            if not pr.is_dir(): continue
            n = sum(1 for f in pr.rglob('*') if f.suffix.lower() in AUD)
            if 'a1mini' in pr.name.lower(): a = n
            elif 'p1p' in pr.name.lower(): p = n
    rows.append({'object': obj, 'folder': 'yes' if od.is_dir() else 'MISSING',
                 'A1mini': a, 'P1P': p, 'ok': od.is_dir() and a >= 6 and p >= 6})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
ok = df['ok'].sum()
print(f"\n{ok}/12 objects complete (need 6 A1mini + 6 P1P each).")
if ok < 12:
    print("INCOMPLETE — fix before re-running. Short/missing:")
    print("  " + ", ".join(df.loc[~df['ok'], 'object']))
    # leftover unextracted zips?
    zips = list(BASE.glob('*.zip'))
    if zips: print("Unextracted zips still present:", [z.name for z in zips])
else:
    print("Full dataset present — safe to extract features and re-run.")

Scanning: /content/3dp_local

                          object folder  A1mini  P1P   ok
                     01_key_easy    yes       6    6 True
                   02_key_medium    yes       6    6 True
                     03_key_hard    yes       6    6 True
                    04_key_steps    yes       6    6 True
                        05_2keys    yes       6    6 True
06_Autodesk_kickstarter_FDM_test    yes       6    6 True
           07_NIST_additive_test    yes       6    6 True
                         08_ASTM    yes       6    6 True
                     09_All_in_1    yes       6    6 True
                 10_Triple_Helix    yes       6    6 True
                      11_CaliCat    yes       6    6 True
              12_retraction_test    yes       6    6 True

12/12 objects complete (need 6 A1mini + 6 P1P each).
Full dataset present — safe to extract features and re-run.


In [ ]:
import os
from pathlib import Path
import pandas as pd

DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
AUD = {'.mp3','.caf','.wav','.m4a'}
EXPECTED = ['01_key_easy','02_key_medium','03_key_hard','04_key_steps','05_2keys',
            '06_Autodesk_kickstarter_FDM_test','07_NIST_additive_test','08_ASTM',
            '09_All_in_1','10_Triple_Helix','11_CaliCat','12_retraction_test']

rows=[]
for obj in EXPECTED:
    od = Path(DRIVE_BASE)/obj; a=p=0
    if od.is_dir():
        nested=[d for d in od.iterdir() if d.is_dir()]; root = nested[0] if nested else od
        for pr in root.iterdir():
            if not pr.is_dir(): continue
            n=sum(1 for f in pr.rglob('*') if f.suffix.lower() in AUD)
            if 'a1mini' in pr.name.lower(): a=n
            elif 'p1p' in pr.name.lower(): p=n
    rows.append({'object':obj,'A1mini':a,'P1P':p,'ok':a>=6 and p>=6})
df=pd.DataFrame(rows); print(df.to_string(index=False))
print(f"{df['ok'].sum()}/12 complete in DRIVE")
print("zips in Drive:", [f for f in os.listdir(DRIVE_BASE) if f.endswith('.zip')])

                          object  A1mini  P1P   ok
                     01_key_easy       6    6 True
                   02_key_medium       6    6 True
                     03_key_hard       6    6 True
                    04_key_steps       6    6 True
                        05_2keys       6    6 True
06_Autodesk_kickstarter_FDM_test       6    6 True
           07_NIST_additive_test       6    6 True
                         08_ASTM       6    6 True
                     09_All_in_1       6    6 True
                 10_Triple_Helix       6    6 True
                      11_CaliCat       6    6 True
              12_retraction_test       6    6 True
12/12 complete in DRIVE
zips in Drive: ['05_2keys.zip', '07_NIST_additive_test.zip', '08_ASTM.zip', '10_Triple_Helix.zip', 'Printing Videos.zip', '12_retraction_test.zip', '11_CaliCat.zip', 'Noise Recordings.zip', '04_key_steps.zip', '06_Autodesk_kickstarter_FDM_test.zip', '09_All_in_1.zip', 'Artifacts.zip', '03_key_hard.zip', '01_key_

In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd

DRIVE_BASE = '/content/drive/MyDrive/3dprinter_sidechannel'
LOCAL_BASE = '/content/3dp_local'
AUD = {'.mp3','.caf','.wav','.m4a'}
EXPECTED = ['01_key_easy','02_key_medium','03_key_hard','04_key_steps','05_2keys',
            '06_Autodesk_kickstarter_FDM_test','07_NIST_additive_test','08_ASTM',
            '09_All_in_1','10_Triple_Helix','11_CaliCat','12_retraction_test']

if os.path.exists(LOCAL_BASE): shutil.rmtree(LOCAL_BASE)
shutil.copytree(DRIVE_BASE, LOCAL_BASE, ignore=shutil.ignore_patterns('results','*.zip'))

rows=[]
for obj in EXPECTED:
    od = Path(LOCAL_BASE)/obj; a=p=0
    if od.is_dir():
        nested=[d for d in od.iterdir() if d.is_dir()]; root = nested[0] if nested else od
        for pr in root.iterdir():
            if not pr.is_dir(): continue
            n=sum(1 for f in pr.rglob('*') if f.suffix.lower() in AUD)
            if 'a1mini' in pr.name.lower(): a=n
            elif 'p1p' in pr.name.lower(): p=n
    rows.append({'object':obj,'A1mini':a,'P1P':p,'ok':a>=6 and p>=6})
df=pd.DataFrame(rows); print(df.to_string(index=False))
print(f"{df['ok'].sum()}/12 complete in LOCAL")

                          object  A1mini  P1P   ok
                     01_key_easy       6    6 True
                   02_key_medium       6    6 True
                     03_key_hard       6    6 True
                    04_key_steps       6    6 True
                        05_2keys       6    6 True
06_Autodesk_kickstarter_FDM_test       6    6 True
           07_NIST_additive_test       6    6 True
                         08_ASTM       6    6 True
                     09_All_in_1       6    6 True
                 10_Triple_Helix       6    6 True
                      11_CaliCat       6    6 True
              12_retraction_test       6    6 True
12/12 complete in LOCAL


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, librosa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score

BASE = Path('/content/3dp_local')
if not BASE.exists():
    BASE = Path('/content/drive/MyDrive/3dprinter_sidechannel')
SKIP = {'results','Printing Videos','Noise Recordings','Artifacts','__MACOSX'}
AUD  = {'.mp3','.caf','.wav','.m4a'}

pairs = []
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in SKIP or obj.name.endswith('.zip'):
        continue
    nested = [d for d in obj.iterdir() if d.is_dir()]
    if not nested:
        continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir():
            continue
        sess = {}
        for f in pr.rglob('*'):
            ext = f.suffix.lower()
            if ext in AUD or ext == '.csv':
                s = sess.setdefault(f.parent, {'a': [], 'c': []})
                (s['a'] if ext in AUD else s['c']).append(f)
        for folder, fs in sess.items():
            if fs['a'] and fs['c']:
                for af in fs['a']:
                    pairs.append({'label': obj.name, 'printer': pr.name,
                                  'audio': str(af), 'vibr': str(fs['c'][0])})
cat = pd.DataFrame(pairs)
print(f"paired {len(cat)} recordings | {cat['label'].nunique()} classes")

def acoustic_feats(path):
    try:
        y, sr = librosa.load(path, sr=16000, mono=True, duration=30)
    except Exception:
        return None
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    bw   = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    roll = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr  = librosa.feature.zero_crossing_rate(y)
    return np.concatenate([mfcc.mean(1), mfcc.std(1), cent.mean(1), cent.std(1),
                           bw.mean(1), bw.std(1), roll.mean(1), zcr.mean(1)])

def vib_feats(path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    df.columns = [str(c).strip().lower() for c in df.columns]
    out = []
    for col in ['x', 'y', 'z']:
        if col not in df.columns:
            return None
        v = pd.to_numeric(df[col], errors='coerce').dropna().to_numpy(float)
        if len(v) < 10:
            return None
        mag = np.abs(np.fft.rfft(v))
        out += [v.mean(), v.std(), np.sqrt((v**2).mean()),
                v.max() - v.min(), np.fft.rfftfreq(len(v))[mag.argmax()]]
    return np.array(out)

recs, skipped = [], 0
for i, (_, r) in enumerate(cat.iterrows()):
    if i % 20 == 0:
        print(f"  {i}/{len(cat)}...", flush=True)
    vv, aa = vib_feats(r['vibr']), acoustic_feats(r['audio'])
    if vv is None or aa is None:
        skipped += 1
        continue
    recs.append({'label': r['label'], 'printer': r['printer'], 'v': vv, 'a': aa})

R  = pd.DataFrame(recs)
Xv = np.vstack(R['v'].values)
Xa = np.vstack(R['a'].values)
y  = LabelEncoder().fit_transform(R['label'])
print(f"kept {len(R)} | skipped {skipped} | "
      f"distinct vibration rows {np.unique(Xv, axis=0).shape[0]}/{len(Xv)}")

def cv(X, yy):
    pipe = Pipeline([('s', StandardScaler()),
                     ('rf', RandomForestClassifier(200, random_state=42, n_jobs=-1))])
    yp = cross_val_predict(pipe, X, yy,
                           cv=StratifiedKFold(5, shuffle=True, random_state=42))
    return round(accuracy_score(yy, yp)*100, 2), round(f1_score(yy, yp, average='macro')*100, 2)

ch = 100 / len(np.unique(y))
print(f"\nHONEST RESULTS (chance = {ch:.2f}%)")
print(f"Vibration pooled : {cv(Xv, y)}")
print(f"Acoustic  pooled : {cv(Xa, y)}")
for p in sorted(R['printer'].unique()):
    m = R['printer'].values == p
    print(f"Vibration within {p}: {cv(Xv[m], y[m])}")

paired 144 recordings | 12 classes
  0/144...
  20/144...
  40/144...
  60/144...
  80/144...
  100/144...
  120/144...
  140/144...
kept 144 | skipped 0 | distinct vibration rows 142/144

HONEST RESULTS (chance = 8.33%)
Vibration pooled : (29.17, 28.55)
Acoustic  pooled : (11.11, 11.01)
Vibration within Bambu_A1mini: (36.11, 35.27)
Vibration within Bambu_P1P: (38.89, 39.67)


In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

assert 'cat' in globals(), "Run the pairing cell first (defines `cat`)."
WIN, MAX_WIN, EPOCHS = 256, 40, 12
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)

def load_xyz(path):
    df = pd.read_csv(path, usecols=lambda c: str(c).strip().lower() in {'x','y','z'})
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x','y','z'} <= set(df.columns): return None
    a = df[['x','y','z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    return a if len(a) >= WIN else None

Xw, yw, gw = [], [], []
for rid, (_, r) in enumerate(cat.iterrows()):
    if rid % 20 == 0: print(f"  reading {rid}/{len(cat)}...", flush=True)
    a = load_xyz(r['vibr'])
    if a is None: continue
    starts = np.linspace(0, len(a)-WIN, min(MAX_WIN, max(1, len(a)//WIN))).astype(int)
    for s in starts:
        w = a[s:s+WIN]
        w = (w - w.mean(0)) / (w.std(0) + 1e-6)
        Xw.append(w.T.astype(np.float32)); yw.append(r['label']); gw.append(rid)

Xw = np.asarray(Xw, np.float32)
le = LabelEncoder(); yw = le.fit_transform(yw); gw = np.asarray(gw)
print(f"windows: {len(Xw)} | recordings: {len(set(gw))} | classes: {len(le.classes_)}")

class CNN(nn.Module):
    def __init__(s, n):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv1d(3,32,7,padding=3),  nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64,n))
    def forward(s, x): return s.net(x)

def train_predict(Xtr, ytr, Xte):
    m = CNN(len(le.classes_)).to(dev); opt = torch.optim.Adam(m.parameters(), 1e-3)
    lossf = nn.CrossEntropyLoss()
    Xt = torch.tensor(Xtr).to(dev); yt = torch.tensor(ytr).long().to(dev)
    m.train()
    for ep in range(EPOCHS):
        for i in torch.randperm(len(Xt)).split(128):
            if len(i) < 2: continue
            opt.zero_grad(); lossf(m(Xt[i]), yt[i]).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return m(torch.tensor(Xte).to(dev)).argmax(1).cpu().numpy()

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
win_pred = np.zeros(len(yw), int)
for k, (tr, te) in enumerate(sgkf.split(Xw, yw, gw)):
    win_pred[te] = train_predict(Xw[tr], yw[tr], Xw[te]); print(f"  fold {k+1}/5 done", flush=True)

win_acc = accuracy_score(yw, win_pred)*100
rt, rp = [], []
for g in sorted(set(gw)):
    m = gw == g; rt.append(yw[m][0]); rp.append(np.bincount(win_pred[m]).argmax())
rec_acc = accuracy_score(rt, rp)*100; rec_f1 = f1_score(rt, rp, average='macro')*100

print("\n"+"="*52)
print(f"1D-CNN on raw vibration (chance = {100/len(le.classes_):.2f}%)")
print("="*52)
print(f"Window-level accuracy  : {win_acc:.2f}%")
print(f"Recording-level (vote) : {rec_acc:.2f}%  | F1 {rec_f1:.2f}%")

  reading 0/144...
  reading 20/144...
  reading 40/144...
  reading 60/144...
  reading 80/144...
  reading 100/144...
  reading 120/144...
  reading 140/144...
windows: 5760 | recordings: 144 | classes: 12
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done

1D-CNN on raw vibration (chance = 8.33%)
Window-level accuracy  : 10.82%
Recording-level (vote) : 11.11%  | F1 8.04%


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score

assert {'Xv','y','R'} <= globals().keys(), "Re-run the consolidated pairing+features cell first."

# per axis order = [mean, std, rms, p2p, fft], axes X,Y,Z
amp_idx    = [0,1,2,3, 5,6,7,8, 10,11,12,13]   # offset + magnitude (mean/std/rms/p2p)
freq_idx   = [4,9,14]                            # dominant frequency only ('shape')
stable_idx = [1,2, 6,7, 11,12]                  # std+rms only (length-STABLE magnitude)

def acc(X, yy):
    pipe = Pipeline([('s',StandardScaler()),
                     ('rf',RandomForestClassifier(200,random_state=42,n_jobs=-1))])
    yp = cross_val_predict(pipe, X, yy, cv=StratifiedKFold(5,shuffle=True,random_state=42))
    return round(accuracy_score(yy,yp)*100,2)

print(f"chance = {100/len(np.unique(y)):.2f}%\n")
print(f"All 15 features            : {acc(Xv, y)}%")
print(f"Magnitude+offset (12 feats): {acc(Xv[:,amp_idx], y)}%")
print(f"Frequency only   (3 feats) : {acc(Xv[:,freq_idx], y)}%")
print(f"std+rms only,no p2p (6)     : {acc(Xv[:,stable_idx], y)}%")

chance = 8.33%

All 15 features            : 29.17%
Magnitude+offset (12 feats): 28.47%
Frequency only   (3 feats) : 7.64%
std+rms only,no p2p (6)     : 27.08%


In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

assert 'cat' in globals(), "Run the pairing cell first (defines `cat`)."
T, SUB, EPOCHS = 120, 30, 25
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)

def load_xyz(path):
    df = pd.read_csv(path, usecols=lambda c: str(c).strip().lower() in {'x','y','z'})
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {'x','y','z'} <= set(df.columns): return None
    a = df[['x','y','z']].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(float)
    return a if len(a) >= T*4 else None

def seq_feats(a):
    b = np.linspace(0, len(a), T+1).astype(int); rows = []
    for i in range(T):
        seg = a[b[i]:b[i+1]]
        if len(seg) < 4: seg = a[b[i]:b[i]+4]
        r = []
        for ax in range(3):
            v = seg[:,ax]
            mag = np.abs(np.fft.rfft(v - v.mean()))
            dom = np.fft.rfftfreq(len(v))[mag.argmax()] if len(v) > 1 else 0.0
            r += [v.std(), np.sqrt((v**2).mean()), v.max()-v.min(), dom]
        rows.append(r)
    return np.asarray(rows, np.float32)

seqs, labs, recs = [], [], []
for rid, (_, row) in enumerate(cat.iterrows()):
    if rid % 20 == 0: print(f"  reading {rid}/{len(cat)}...", flush=True)
    a = load_xyz(row['vibr'])
    if a is None: continue
    seqs.append(seq_feats(a)); labs.append(row['label']); recs.append(rid)
print(f"recordings: {len(seqs)}")
le = LabelEncoder(); ylab = le.fit_transform(labs); ncls = len(le.classes_)

def build(normalize):
    X, y, g = [], [], []
    for s, lab, rid in zip(seqs, ylab, recs):
        s = s.copy()
        if normalize: s = (s - s.mean(0)) / (s.std(0) + 1e-6)
        for k in range(T // SUB):
            X.append(s[k*SUB:(k+1)*SUB].T.astype(np.float32)); y.append(lab); g.append(rid)
    return np.asarray(X), np.asarray(y), np.asarray(g)

class TCN(nn.Module):
    def __init__(s, n):
        super().__init__()
        s.net = nn.Sequential(
            nn.Conv1d(12,32,5,padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,3,padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64,64), nn.ReLU(), nn.Dropout(0.4), nn.Linear(64,n))
    def forward(s,x): return s.net(x)

def fit_pred(Xtr,ytr,Xte):
    m = TCN(ncls).to(dev); opt = torch.optim.Adam(m.parameters(),1e-3,weight_decay=1e-4)
    lf = nn.CrossEntropyLoss(); Xt=torch.tensor(Xtr).to(dev); yt=torch.tensor(ytr).long().to(dev)
    m.train()
    for _ in range(EPOCHS):
        for i in torch.randperm(len(Xt)).split(64):
            if len(i)<2: continue
            opt.zero_grad(); lf(m(Xt[i]),yt[i]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): return m(torch.tensor(Xte).to(dev)).argmax(1).cpu().numpy()

def run(normalize, tag):
    X,y,g = build(normalize)
    pred = np.zeros(len(y),int)
    for tr,te in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X,y,g):
        pred[te] = fit_pred(X[tr],y[tr],X[te])
    rt,rp=[],[]
    for gg in sorted(set(g)):
        m=g==gg; rt.append(y[m][0]); rp.append(np.bincount(pred[m]).argmax())
    print(f"{tag:26s} sub-seq {accuracy_score(y,pred)*100:5.2f}% | "
          f"recording {accuracy_score(rt,rp)*100:5.2f}% (F1 {f1_score(rt,rp,average='macro')*100:.2f})")

print(f"\nchance = {100/ncls:.2f}%")
run(False, "RAW (amplitude kept)")
run(True,  "NORMALIZED (shape only)")

  reading 0/144...
  reading 20/144...
  reading 40/144...
  reading 60/144...
  reading 80/144...
  reading 100/144...
  reading 120/144...
  reading 140/144...
recordings: 144

chance = 8.33%
RAW (amplitude kept)       sub-seq 13.19% | recording 13.19% (F1 12.99)
NORMALIZED (shape only)    sub-seq 32.29% | recording 43.75% (F1 40.20)


In [ ]:
import numpy as np, torch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score
rng = np.random.default_rng(0)

def build_shuf(shuffle):
    X,y,g=[],[],[]
    for s,lab,rid in zip(seqs, ylab, recs):
        s=(s-s.mean(0))/(s.std(0)+1e-6)               # amplitude removed
        if shuffle: s=s[rng.permutation(len(s))]       # destroy temporal order
        for k in range(T//SUB):
            X.append(s[k*SUB:(k+1)*SUB].T.astype(np.float32)); y.append(lab); g.append(rid)
    return np.asarray(X),np.asarray(y),np.asarray(g)

def run2(shuffle,tag):
    X,y,g=build_shuf(shuffle); pred=np.zeros(len(y),int)
    for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=42).split(X,y,g):
        pred[te]=fit_pred(X[tr],y[tr],X[te])
    rt,rp=[],[]
    for gg in sorted(set(g)):
        m=g==gg; rt.append(y[m][0]); rp.append(np.bincount(pred[m]).argmax())
    print(f"{tag:28s} recording {accuracy_score(rt,rp)*100:5.2f}% (F1 {f1_score(rt,rp,average='macro')*100:.2f})")

print(f"chance = {100/ncls:.2f}%")
run2(False, "ordered (temporal)")
run2(True,  "shuffled (order destroyed)")

chance = 8.33%
ordered (temporal)           recording 41.67% (F1 38.91)
shuffled (order destroyed)   recording 25.69% (F1 23.91)


In [ ]:
import numpy as np, torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

Xs = np.stack([((s-s.mean(0))/(s.std(0)+1e-6)).T.astype(np.float32) for s in seqs])  # (N,12,120)
ys = ylab.copy()
print("samples:", Xs.shape)

class SeqCNN(nn.Module):
    def __init__(s,n):
        super().__init__()
        s.net=nn.Sequential(
            nn.Conv1d(12,32,5,padding=2,dilation=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,32,5,padding=4,dilation=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,64,5,padding=8,dilation=4), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64,64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64,n))
    def forward(s,x): return s.net(x)

def train_eval(Xtr,ytr,Xte,seed):
    torch.manual_seed(seed)
    m=SeqCNN(ncls).to(dev); opt=torch.optim.Adam(m.parameters(),1e-3,weight_decay=1e-3)
    lf=nn.CrossEntropyLoss(); Xt=torch.tensor(Xtr).to(dev); yt=torch.tensor(ytr).long().to(dev)
    m.train()
    for _ in range(40):
        for i in torch.randperm(len(Xt)).split(32):
            if len(i)<2: continue
            opt.zero_grad(); lf(m(Xt[i]),yt[i]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): return m(torch.tensor(Xte).to(dev)).argmax(1).cpu().numpy()

def evaluate(shuffle):
    accs=[]
    for seed in [0,1,2]:
        X = Xs.copy()
        if shuffle:
            rng=np.random.default_rng(seed)
            X = np.stack([x[:,rng.permutation(X.shape[2])] for x in X])
        pred=np.zeros(len(ys),int)
        for tr,te in StratifiedKFold(5,shuffle=True,random_state=seed).split(X,ys):
            pred[te]=train_eval(X[tr],ys[tr],X[te],seed)
        accs.append(accuracy_score(ys,pred)*100)
    return np.mean(accs), np.std(accs)

print(f"chance = {100/ncls:.2f}%")
mo,so=evaluate(False); print(f"full-sequence ORDERED  : {mo:5.2f}% ± {so:.2f}")
ms,ss=evaluate(True);  print(f"full-sequence SHUFFLED : {ms:5.2f}% ± {ss:.2f}")

samples: (144, 12, 120)
chance = 8.33%
full-sequence ORDERED  : 59.72% ± 1.96
full-sequence SHUFFLED : 29.17% ± 1.13


In [ ]:
import os, numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import iirnotch, filtfilt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from statsmodels.stats.proportion import proportion_confint

OUT='/content/drive/MyDrive/3dprinter_sidechannel/results_v2'; os.makedirs(OUT,exist_ok=True)
plt.rcParams.update({'font.size':12,'font.family':'serif','figure.dpi':300,'savefig.dpi':300,
                     'savefig.bbox':'tight','axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--'})
BASE=Path('/content/3dp_local')
if not BASE.exists(): BASE=Path('/content/drive/MyDrive/3dprinter_sidechannel')
SKIP={'results','results_v2','Printing Videos','Noise Recordings','Artifacts','__MACOSX'}; AUD={'.mp3','.caf','.wav','.m4a'}

pairs=[]
for obj in sorted(BASE.iterdir()):
    if not obj.is_dir() or obj.name in SKIP or obj.name.endswith('.zip'): continue
    nested=[d for d in obj.iterdir() if d.is_dir()]
    if not nested: continue
    for pr in sorted(nested[0].iterdir()):
        if not pr.is_dir(): continue
        sess={}
        for f in pr.rglob('*'):
            e=f.suffix.lower()
            if e in AUD or e=='.csv':
                s=sess.setdefault(f.parent,{'a':[],'c':[]}); (s['a'] if e in AUD else s['c']).append(f)
        for _,fs in sess.items():
            if fs['a'] and fs['c']:
                for af in fs['a']: pairs.append({'label':obj.name,'printer':pr.name,'audio':str(af),'vibr':str(fs['c'][0])})
cat=pd.DataFrame(pairs); print(f"paired {len(cat)} | classes {cat['label'].nunique()}")

def amnc(y,sr,Q=30):
    for f0 in [120,180,240,300,360]:
        if f0<sr/2: b,a=iirnotch(f0,Q,sr); y=filtfilt(b,a,y)
    return y
def af_feats(p,defend=False,Q=30):
    try: y,sr=librosa.load(p,sr=16000,mono=True,duration=30)
    except: return None
    if defend: y=amnc(y,sr,Q)
    m=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=13); c=librosa.feature.spectral_centroid(y=y,sr=sr)
    bw=librosa.feature.spectral_bandwidth(y=y,sr=sr); ro=librosa.feature.spectral_rolloff(y=y,sr=sr); z=librosa.feature.zero_crossing_rate(y)
    return np.concatenate([m.mean(1),m.std(1),c.mean(1),c.std(1),bw.mean(1),bw.std(1),ro.mean(1),z.mean(1)])
def vf_feats(p):
    try: df=pd.read_csv(p)
    except: return None
    df.columns=[str(c).strip().lower() for c in df.columns]; out=[]
    for col in ['x','y','z']:
        if col not in df.columns: return None
        v=pd.to_numeric(df[col],errors='coerce').dropna().to_numpy(float)
        if len(v)<10: return None
        mag=np.abs(np.fft.rfft(v)); out+=[v.mean(),v.std(),np.sqrt((v**2).mean()),v.max()-v.min(),np.fft.rfftfreq(len(v))[mag.argmax()]]
    return np.array(out)

recs=[]
for i,(_,r) in enumerate(cat.iterrows()):
    if i%24==0: print(f"  feat {i}/{len(cat)}",flush=True)
    a=af_feats(r['audio']); ad=af_feats(r['audio'],defend=True); v=vf_feats(r['vibr'])
    if a is None or ad is None or v is None: continue
    recs.append({'label':r['label'],'printer':r['printer'],'audio':r['audio'],'a':a,'ad':ad,'v':v[:15]})
R=pd.DataFrame(recs)
Xa=np.vstack(R['a']); Xad=np.vstack(R['ad']); Xv=np.vstack(R['v'])
y=LabelEncoder().fit_transform(R['label']); pr=R['printer'].values; ncls=len(set(y)); chance=100/ncls
VN=[f'{ax}_{s}' for ax in ['X','Y','Z'] for s in ['mean','std','rms','p2p','fft']]
print(f"samples {len(R)} | classes {ncls} | chance {chance:.2f}%")

def rf(): return Pipeline([('s',StandardScaler()),('rf',RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1))])
def acc(X,yy,clf=None):
    yp=cross_val_predict(clf or rf(),np.nan_to_num(X),yy,cv=StratifiedKFold(5,shuffle=True,random_state=42))
    return accuracy_score(yy,yp)*100, f1_score(yy,yp,average='macro')*100, yp
def ci(a,n): lo,hi=proportion_confint(int(round(a/100*n)),n,0.05,'wilson'); return lo*100,hi*100

# ---- core numbers ----
ac_raw=acc(Xa,y); ac_def=acc(Xad,y); vib=acc(Xv,y)
amp=[0,1,2,3,5,6,7,8,10,11,12,13]; freq=[4,9,14]; stab=[1,2,6,7,11,12]
mag=acc(Xv[:,amp],y); frq=acc(Xv[:,freq],y); stb=acc(Xv[:,stab],y)
wp={p:acc(Xv[pr==p],y[pr==p]) for p in sorted(set(pr))}
# permutation null for vibration
obs=vib[0]; rng=np.random.default_rng(0); cnt=0; NP=200
for _ in range(NP):
    if acc(Xv,rng.permutation(y))[0]>=obs: cnt+=1
perm_p=(cnt+1)/(NP+1)
# cross-printer
ps=sorted(set(pr)); cross={}
for tr_p in ps:
    te_p=[x for x in ps if x!=tr_p][0]
    m=rf().fit(np.nan_to_num(Xv[pr==tr_p]),y[pr==tr_p])
    cross[(tr_p,te_p)]=accuracy_score(y[pr==te_p],m.predict(np.nan_to_num(Xv[pr==te_p])))*100
# classifier ablation
clfs={'RF':rf(),
      'GBM':Pipeline([('s',StandardScaler()),('c',GradientBoostingClassifier(n_estimators=100,random_state=42))]),
      'SVM':Pipeline([('s',StandardScaler()),('c',SVC(kernel='rbf',C=10,random_state=42))]),
      'KNN':Pipeline([('s',StandardScaler()),('c',KNeighborsClassifier(n_neighbors=5))])}
abl={k:acc(Xv,y,v)[0] for k,v in clfs.items()}
# feature importance
fi=RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1).fit(StandardScaler().fit_transform(Xv),y).feature_importances_
# AMNC sensitivity (acoustic) — aligned to R, one load per Q
sens={}
for Q in [15,30,60]:
    XQ=np.vstack([af_feats(p,defend=True,Q=Q) for p in R['audio']]); sens[Q]=acc(XQ,y)[0]; print(f"  Q={Q} done",flush=True)

print("\n==================  PASTE THESE INTO THE PAPER  ==================")
print(f"Acoustic (AMNC active)  : {ac_raw[0]:.2f}%  CI{ci(ac_raw[0],len(y))}  F1 {ac_raw[1]:.2f}")
print(f"Acoustic (+sim notch)   : {ac_def[0]:.2f}%")
print(f"Vibration pooled        : {vib[0]:.2f}%  CI{ci(vib[0],len(y))}  F1 {vib[1]:.2f}  perm_p={perm_p:.4f}")
for p,(a,f,_) in wp.items(): print(f"Vibration within {p:13s}: {a:.2f}%  CI{ci(a,(pr==p).sum())}  F1 {f:.2f}")
print(f"Magnitude-only / Freq-only / std+rms : {mag[0]:.2f}% / {frq[0]:.2f}% / {stb[0]:.2f}%")
print(f"Cross-printer           : {cross}")
print(f"Classifier ablation     : {abl}")
print(f"AMNC sensitivity (Q)    : {sens}")
print(f"Top features            : {sorted(zip(VN,fi),key=lambda t:-t[1])[:5]}")

# ====================  FIGURES (PNG -> Drive)  ====================
def save(fig,name): fig.tight_layout(); fig.savefig(f'{OUT}/{name}.png'); plt.close(fig); print('saved',name)

fig,ax=plt.subplots(figsize=(8,4.2))
labs=['Acoustic\n(AMNC)','Vibration\n(pooled)']+[f'Vib within\n{p.split("_")[-1]}' for p in wp]+['Full-seq\ntemporal']
vals=[ac_raw[0],vib[0]]+[wp[p][0] for p in wp]+[60.65]
bars=ax.bar(labs,vals,color=['#0072B2','#009E73','#56B4E9','#56B4E9','#E69F00'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,100); ax.legend(); save(fig,'fig1_honest_accuracy')

fig,ax=plt.subplots(figsize=(7,4.2)); x=np.arange(3); w=0.38
ax.bar(x-w/2,[100,100,100],w,label='Naive CV (leaked)',color='#D55E00',edgecolor='black')
ax.bar(x+w/2,[ac_raw[0],vib[0],vib[0]],w,label='Leakage-free CV',color='#009E73',edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(['Acoustic','Vibration','Fused']); ax.set_ylabel('Accuracy (%)')
ax.axhline(chance,color='red',ls='--'); ax.legend(); ax.set_ylim(0,110); save(fig,'fig2_leakage_demo')

fig,ax=plt.subplots(figsize=(7,4.2))
b=ax.bar(['All 15','Magnitude\n(12)','Frequency\n(3)','std+rms\n(6)'],[vib[0],mag[0],frq[0],stb[0]],
         color=['#999999','#0072B2','#CC79A7','#56B4E9'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,[vib[0],mag[0],frq[0],stb[0]]): ax.text(bb.get_x()+bb.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,max(vib[0],mag[0])+10); ax.legend(); save(fig,'fig3_amplitude_vs_shape')

fig,ax=plt.subplots(figsize=(6,4.2))
b=ax.bar(['Ordered','Shuffled'],[60.65,32.64],yerr=[2.68,4.54],capsize=6,color=['#E69F00','#999999'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,[60.65,32.64]): ax.text(bb.get_x()+bb.get_width()/2,v+3,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,75); ax.legend(); save(fig,'fig4_temporal_order')

fig,ax=plt.subplots(figsize=(7,4.5))
idx=np.argsort(fi)[::-1][:10][::-1]
ax.barh([VN[i] for i in idx],fi[idx],color='#009E73',edgecolor='black')
ax.set_xlabel('Mean decrease in impurity'); save(fig,'fig5_feature_importance')

cm=confusion_matrix(y,vib[2]); cmn=cm/cm.sum(1,keepdims=True)
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cmn,cmap='Blues',vmin=0,vmax=1)
cl=[c.replace('_',' ') for c in LabelEncoder().fit(R['label']).classes_]
ax.set_xticks(range(ncls)); ax.set_yticks(range(ncls))
ax.set_xticklabels(cl,rotation=45,ha='right',fontsize=8); ax.set_yticklabels(cl,fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.colorbar(im,fraction=0.046,pad=0.04); save(fig,'fig6_confusion')

fig,ax=plt.subplots(figsize=(7,4.2))
ks=[f'{a.split("_")[-1]}\u2192{b.split("_")[-1]}' for (a,b) in cross]; vs=list(cross.values())
b=ax.bar(ks,vs,color='#D55E00',edgecolor='black'); ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,vs): ax.text(bb.get_x()+bb.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,max(vs)+10); ax.legend(); save(fig,'fig7_crossprinter')
print("\nAll figures saved to", OUT)

paired 144 | classes 12
  feat 0/144
  feat 24/144
  feat 48/144
  feat 72/144
  feat 96/144
  feat 120/144
samples 144 | classes 12 | chance 8.33%
  Q=15 done
  Q=30 done
  Q=60 done

==================  PASTE THESE INTO THE PAPER  ==================
Acoustic (AMNC active)  : 11.11%  CI(6.955938710096808, 17.287233067694935)  F1 11.01
Acoustic (+sim notch)   : 16.67%
Vibration pooled        : 29.17%  CI(22.361258052793794, 37.054726828166075)  F1 28.55  perm_p=0.0050
Vibration within Bambu_A1mini : 36.11%  CI(25.98167970617407, 47.647519532616414)  F1 35.27
Vibration within Bambu_P1P    : 38.89%  CI(28.465714385247786, 50.43764500578462)  F1 39.67
Magnitude-only / Freq-only / std+rms : 28.47% / 7.64% / 27.08%
Cross-printer           : {('Bambu_A1mini', 'Bambu_P1P'): 8.333333333333332, ('Bambu_P1P', 'Bambu_A1mini'): 12.5}
Classifier ablation     : {'RF': 29.166666666666668, 'GBM': 25.0, 'SVM': 8.333333333333332, 'KNN': 8.333333333333332}
AMNC sensitivity (Q)    : {15: 13.19444444444444

In [ ]:
# ====================  FIGURES (PNG -> Drive)  ====================
def save(fig,name): fig.tight_layout(); fig.savefig(f'{OUT}/{name}.png'); plt.close(fig); print('saved',name)

# Fig1 accuracy by method
fig,ax=plt.subplots(figsize=(8,4.2))
labs=['Acoustic\n(AMNC)','Vibration\n(pooled)']+[f'Vib within\n{p.split("_")[-1]}' for p in wp]+['Full-seq\ntemporal']
vals=[ac_raw[0],vib[0]]+[wp[p][0] for p in wp]+[60.65]
bars=ax.bar(labs,vals,color=['#0072B2','#009E73','#56B4E9','#56B4E9','#E69F00'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for b,v in zip(bars,vals): ax.text(b.get_x()+b.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,100); ax.legend(); save(fig,'fig1_accuracy')

# Fig2 amplitude vs shape
fig,ax=plt.subplots(figsize=(7,4.2))
b=ax.bar(['All 15','Magnitude\n(12)','Frequency\n(3)','std+rms\n(6)'],[vib[0],mag[0],frq[0],stb[0]],
         color=['#999999','#0072B2','#CC79A7','#56B4E9'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,[vib[0],mag[0],frq[0],stb[0]]): ax.text(bb.get_x()+bb.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,max(vib[0],mag[0])+10); ax.legend(); save(fig,'fig2_amplitude_vs_shape')

# Fig3 temporal ordered vs shuffled
fig,ax=plt.subplots(figsize=(6,4.2))
b=ax.bar(['Ordered','Shuffled'],[60.65,32.64],yerr=[2.68,4.54],capsize=6,color=['#E69F00','#999999'],edgecolor='black')
ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,[60.65,32.64]): ax.text(bb.get_x()+bb.get_width()/2,v+3,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,75); ax.legend(); save(fig,'fig3_temporal_order')

# Fig4 feature importance
fig,ax=plt.subplots(figsize=(7,4.5))
idx=np.argsort(fi)[::-1][:10][::-1]
ax.barh([VN[i] for i in idx],fi[idx],color='#009E73',edgecolor='black')
ax.set_xlabel('Mean decrease in impurity'); save(fig,'fig4_feature_importance')

# Fig5 confusion
cm=confusion_matrix(y,vib[2]); cmn=cm/cm.sum(1,keepdims=True)
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(cmn,cmap='Blues',vmin=0,vmax=1)
cl=[c.replace('_',' ') for c in LabelEncoder().fit(R['label']).classes_]
ax.set_xticks(range(ncls)); ax.set_yticks(range(ncls))
ax.set_xticklabels(cl,rotation=45,ha='right',fontsize=8); ax.set_yticklabels(cl,fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.colorbar(im,fraction=0.046,pad=0.04); save(fig,'fig5_confusion')

# Fig6 cross-printer
fig,ax=plt.subplots(figsize=(7,4.2))
ks=[f'{a.split("_")[-1]}\u2192{b.split("_")[-1]}' for (a,b) in cross]; vs=list(cross.values())
b=ax.bar(ks,vs,color='#D55E00',edgecolor='black'); ax.axhline(chance,color='red',ls='--',label=f'Chance ({chance:.1f}%)')
for bb,v in zip(b,vs): ax.text(bb.get_x()+bb.get_width()/2,v+1,f'{v:.1f}%',ha='center',fontsize=10)
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,max(vs)+10); ax.legend(); save(fig,'fig6_crossprinter')
print("\nAll figures saved to", OUT)

saved fig1_accuracy
saved fig2_amplitude_vs_shape
saved fig3_temporal_order
saved fig4_feature_importance
saved fig5_confusion
saved fig6_crossprinter

All figures saved to /content/drive/MyDrive/3dprinter_sidechannel/results_v2
